# Answer a manager's questions about June 2024

*Data Analysis in Python · Lesson 2 final exercise*

A message from your manager on Monday morning:

> Can you get me a few numbers on June before the 10 o'clock? Our five best paid search rows by revenue, how many rows in the export have no campaign at all, what share of June's sessions came from email, and the smallest revenue figure in there. Someone said it was negative, which can't be right?

Answer from June 2024's sessions export, `sessions_export_2024_06.csv`, in the program's data store: `https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/`.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

Add your cells between this one and the **Check your work** cell.

1. Add a parameters cell with the file's URL, and read the file with its dates as dates.
2. Create `june`: the rows dated in June 2024. Every answer below is about these rows only.
3. Create `top_paid_search`: the five paid search rows (medium `cpc`) with the highest revenue, highest first, with the columns `date`, `session_campaign` and `revenue`.
4. Create `no_campaign`: the number of rows whose `session_campaign` is missing (`NaN`).
5. Create `email_share`: email's share of June's sessions (email sessions divided by all sessions), rounded to 4 decimals.
6. Create `smallest_revenue`: the smallest revenue in a single row. Add a text cell that answers your manager: is a negative figure an error?
7. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `june` has 1,790 rows. `top_paid_search` has five rows, with revenue from $686.86 down to $472.92. `no_campaign` is `62`, `email_share` is `0.0913`, and `smallest_revenue` is `-541.91`.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    june = found.get("june")
    if not isinstance(june, pd.DataFrame):
        results.append((False, "june isn't a DataFrame yet. Read the file with parse_dates=[\"date\"], then keep the rows dated in June."))
    elif not pd.api.types.is_datetime64_any_dtype(june["date"]):
        results.append((False, "june's date column is text. Read the file with parse_dates=[\"date\"]."))
    elif len(june) == 1790:
        results.append((True, "june has the 1,790 rows dated in June."))
    elif len(june) == 1933:
        results.append((False, "june has all 1,933 rows of the file, including July 1 and 2. Keep the rows dated on or before \"2024-06-30\"."))
    elif len(june) == 1722:
        results.append((False, "june has 1,722 rows: June 30 is missing. Use <= \"2024-06-30\", or < \"2024-07-01\"."))
    else:
        results.append((False, f"june has {len(june)} rows, but June has 1,790."))

    top = found.get("top_paid_search")
    expected_top = [686.86, 583.15, 492.66, 491.91, 472.92]
    if not isinstance(top, pd.DataFrame) or "revenue" not in top.columns:
        results.append((False, "top_paid_search should be a DataFrame with the columns date, session_campaign and revenue."))
    else:
        revenue = [round(float(value), 2) for value in top["revenue"]]
        if revenue == expected_top and top.columns.tolist() == ["date", "session_campaign", "revenue"] and set(top["session_campaign"]) <= {"2024_us_search_shopping", "2024_us_search_generic"}:
            results.append((True, "top_paid_search has June's five best paid search rows."))
        elif revenue == expected_top:
            results.append((False, f"top_paid_search has the right rows, but its columns are {top.columns.tolist()}. Keep [\"date\", \"session_campaign\", \"revenue\"]."))
        elif sorted(revenue, reverse=True) == expected_top:
            results.append((False, "top_paid_search has the right rows, but not highest first. Sort with ascending=False."))
        else:
            results.append((False, f"top_paid_search's revenue should be {expected_top}, but it is {revenue}. Keep june's rows whose session_medium is cpc, then sort by revenue, highest first."))

    no_campaign = found.get("no_campaign")
    if no_campaign == 62:
        results.append((True, "62 of June's rows have no campaign (NaN). (not set) and - are lost campaigns too: lesson 3 counts them."))
    elif no_campaign == 68:
        results.append((False, "no_campaign is 68, which counts July 1 and 2 too. Count in june, not in the whole file."))
    elif no_campaign == 1790 - 62:
        results.append((False, "no_campaign is the number of rows that have a campaign. Count the missing ones: june[\"session_campaign\"].isna().sum()."))
    else:
        results.append((False, f"no_campaign should be june[\"session_campaign\"].isna().sum(). It is {no_campaign!r}."))

    share = found.get("email_share")
    if pd.api.types.is_number(share) and abs(share - 0.0913) < 0.00005:
        results.append((True, "Email brought 9.13% of June's sessions."))
    elif pd.api.types.is_number(share) and abs(share - 0.0957) < 0.00005:
        results.append((False, "email_share is 0.0957, which includes July 1 and 2. Use june's rows for both totals."))
    elif pd.api.types.is_number(share) and abs(share - 0.0726) < 0.00005:
        results.append((False, "email_share is 0.0726: that's the share of rows, not of sessions. Divide email's total sessions by june's total sessions."))
    elif pd.api.types.is_number(share) and abs(share - 9.13) < 0.005:
        results.append((False, "email_share is a percentage. Keep it as a share between 0 and 1: 0.0913."))
    else:
        results.append((False, f"email_share should be email's sessions divided by all of june's sessions, rounded to 4 decimals. It is {share!r}."))

    smallest = found.get("smallest_revenue")
    if pd.api.types.is_number(smallest) and abs(smallest + 541.91) < 0.005:
        results.append((True, "The smallest revenue is -$541.91: a refund, exported as a negative row. Not an error, but anyone who totals revenue needs to know."))
    else:
        results.append((False, f"smallest_revenue should be june[\"revenue\"].min(). It is {smallest!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. These are numbers you can send.")


check_your_work()

<details>
<summary><b>Show hint</b></summary>

Profile the file before you answer anything (topic 2.2.): `tail()` and the date column's `max()` show that the export runs into July. Keep June's rows with a date condition (topic 2.3.) and use `june` for every answer after that. The top five is a filter, then a sort with `ascending=False`, then `head(5)` and the three columns. `no_campaign` is `isna().sum()`. For `email_share`, total the sessions of the email rows and divide by the total sessions of all of `june`'s rows: sessions, not rows.

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
import pandas as pd

FILE_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_06.csv"
export = pd.read_csv(FILE_URL, parse_dates=["date"])
export["date"].max()
```

```python
june = export[export["date"] <= "2024-06-30"]
len(june)
```

```python
top_paid_search = june[june["session_medium"] == "cpc"].sort_values("revenue", ascending=False).head(5)
top_paid_search = top_paid_search.loc[:, ["date", "session_campaign", "revenue"]]
top_paid_search
```

```python
no_campaign = june["session_campaign"].isna().sum()
email_sessions = june.loc[june["session_medium"] == "email", "sessions"].sum()
email_share = round(email_sessions / june["sessions"].sum(), 4)
smallest_revenue = june["revenue"].min()
no_campaign, email_share, smallest_revenue
```

</details>